# اليوم 4، المعمل 7: الضبط وإغلاق الحلقة

نضبط النموذج داخل train باستخدام طيات. بعد تثبيت كل قرار، نفتح test مرة واحدة ونكتب بطاقة النموذج.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "raw"
RANDOM_STATE = 42


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from scipy.stats import randint, loguniform
from manafeth.data import load_customers, split_customers
from manafeth.features import build_preprocessor
from manafeth.evaluation import threshold_for_fraction

df = load_customers(DATA)
X_train, X_test, y_train, y_test = split_customers(df)
pipe = Pipeline([("prep", build_preprocessor(scale_numeric=False)), ("model", RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))])


## بحث محدود

استخدم 12 تجربة فقط. البحث ليس مسابقة استهلاك حوسبة. دوّن مساحة البحث والميزانية قبل التشغيل.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
params = {
    "model__n_estimators": randint(150, 450),
    "model__max_depth": randint(5, 16),
    "model__min_samples_leaf": randint(5, 35),
    "model__max_features": ["sqrt", .6, .9],
}
search = RandomizedSearchCV(pipe, params, n_iter=12, scoring="average_precision", cv=cv, n_jobs=-1, random_state=RANDOM_STATE, refit=True, verbose=1)
search.fit(X_train, y_train)
print("best CV PR-AUC:", round(search.best_score_, 3))
print(search.best_params_)

# The test set is opened once, after the full protocol is frozen.
test_probability = search.best_estimator_.predict_proba(X_test)[:, 1]
threshold = threshold_for_fraction(test_probability, .20)
test_prediction = test_probability >= threshold
print("test PR-AUC:", round(average_precision_score(y_test, test_probability), 3))
print("threshold:", round(threshold, 3))
print("precision:", round(precision_score(y_test, test_prediction), 3))
print("recall:", round(recall_score(y_test, test_prediction), 3))


## الاختبار النهائي

بعد اختيار الإعدادات، استخدم best_estimator_ على test مرة واحدة. احسب PR-AUC وحدد عتبة أعلى 20% ثم Precision وRecall.

In [ ]:
# أُنجزت خطوات هذا القسم في خلية الحل السابقة.


## بطاقة النموذج

انسخ `templates/MODEL_CARD.md` إلى `artifacts/MODEL_CARD.md`. سجّل intended use، البيانات، CV، test، الشرائح الضعيفة، وحدود الاستخدام.

**ناتج التسليم:** أفضل إعدادات، نتائج CV وtest، عتبة القرار، وبطاقة نموذج مكتملة.